In [6]:
import cv2
import importlib
import numpy as np
import math

In [7]:
video_paths = [
    "./dataset/videos/La_Chute_dune_Plume.mp4",
    "./dataset/videos/tears_of_steel_1080p.mp4",
    "./dataset/videos/sintel-1024-surround.mp4",
]
gt_files = [
    "./dataset/videos/La_Chute_dune_Plume_scenes.txt",
    "./dataset/videos/tears_of_steel_1080p_scenes.txt",
    "./dataset/videos/sintel-1024-surround_scenes.txt",
]

THRESH = 0.5
MIN_LEN = 10
H_BINS = 18
S_BINS = 8
V_BINS = 8

In [ ]:
video_path = "./dataset/videos/tears_of_steel_1080p.mp4"
THRESH = 0.5
MIN_LEN = 10

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
print("fps:", fps)

prev_hsv = None
cuts = [0]
frame_idx = 0
maxDiff = 0
minDiff = float('inf')

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame =cv2.resize(frame, (160, 160))
    hsv =cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    if prev_hsv is not None:
        diff= np.abs(hsv.astype("int32") - prev_hsv.astype("int32")).mean()
        if diff >THRESH:
            cuts.append(frame_idx)
        if diff> maxDiff:
            maxDiff = diff
        if diff <minDiff:
            minDiff = diff
    prev_hsv = hsv.copy()
    frame_idx += 1

cap.release()

cuts.append(frame_idx)

cleaned_cuts = [cuts[0]]
for i in range(1, len(cuts)):
    start = cuts[i-1]
    end = cuts[i]
    if (end - start) < MIN_LEN:
        cleaned_cuts.append(end)

fps: 24.0


In [17]:
print("max diff:", maxDiff)
print("min diff:", minDiff)
print("total frames:", len(cleaned_cuts))

max diff: 87.25713541666667
min diff: 0.0
total frames: 17099


In [ ]:
import importlib
import Expirements.SceneSegmentation.utils
importlib.reload(Expirements.SceneSegmentation.utils)
from Expirements.SceneSegmentation.utils import bgr_to_hsv, calc_hist

H_BINS = 18
S_BINS = 8
V_BINS = 8
FEAT_DIM = H_BINS + S_BINS + V_BINS

cap2 = cv2.VideoCapture(video_path)
all_feats = []

for idx in cleaned_cuts:
    cap2.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frm = cap2.read()
    if not ok:
        continue

    hsv_f= cv2.cvtColor(frm, cv2.COLOR_BGR2HSV)
    h_hist =cv2.calcHist([hsv_f], [0], None, [H_BINS], [0, 180])
    s_hist=cv2.calcHist([hsv_f], [1], None, [S_BINS], [0, 256])
    v_hist= cv2.calcHist([hsv_f], [2], None, [V_BINS], [0, 256])

    feat = np.concatenate([h_hist,s_hist, v_hist])
    feat = feat/(feat.sum() + 1e-9)
    all_feats.append(feat.astype(np.float32))

cap2.release()
feats = np.array(all_feats)
N = len(feats)
print("feats shape", feats.shape)

KeyboardInterrupt: 

In [ ]:
# distance between each pair of shots feature vectors 
dist_mat = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        dist_mat[i, j]=np.sqrt(np.sum((feats[i] - feats[j])**2))

dist_mat = (dist_mat + dist_mat.T) / 2.0

print("dist_mat shape", dist_mat.shape)

dist_mat shape (0, 0)


In [ ]:
# get K number of scenes using SVD elbow
sv = np.linalg.svd(dist_mat, compute_uv=False)

print ("singular values:", sv[:10])

sv_log = np.log(sv[:int(len(sv)*0.5)] + 1e-9)
print("log singular values:", sv_log[:10])

# y = mx + c
constant_term=sv_log[0]
slope = (sv_log[-1] - sv_log[0]) / (((len(sv_log) - 1) - 0) + 1e-9)

max_d = 0
K_est = 2
for idx in range(len(sv_log)):
    pt = np.array([float(idx), sv_log[idx]])
    # using formula for perpendicular distance from point to line
    d = abs((-sv_log[idx] + constant_term + slope * idx)) / math.sqrt(slope**2 + 1**2)
    print(f"dist {idx}: {round(d, 4)}")
    if d > max_d:
        max_d = d
        K_est = idx

K_est = max(K_est, 2)  
K_est = min(K_est, N - 1)
print(f"estimated K = {K_est}")

singular values: []
log singular values: []


IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
_area_sums_memo = {}

def area_sums(parent_size, num_parts):
    key = (parent_size, num_parts)
    if key in _area_sums_memo:
        return _area_sums_memo[key]
    if num_parts == 0:
        result = frozenset({0})
    elif num_parts == 1:
        result = frozenset({parent_size * parent_size})
    elif parent_size < num_parts:
        result = frozenset()
    else:
        res = set()
        for i in range(int(round(parent_size / 2))):
            sq = (i + 1) * (i + 1)
            for s in area_sums(parent_size - i - 1, num_parts - 1):
                res.add(sq + s)
        result = frozenset(res)
    _area_sums_memo[key] = result
    return result

In [ ]:
# 2D prefix sum for O(1) submatrix sum queries
psum = np.zeros((N+1, N+1))
for ii in range(1, N+1):
    for jj in range(1, N+1):
        psum[ii][jj] = dist_mat[ii-1][jj-1] + psum[ii-1][jj] + psum[ii][jj-1] - psum[ii-1][jj-1]

def dsum(a, b):
    # sum of dist_mat[a:b+1, a:b+1], a and b are 0-indexed inclusive
    return psum[b+1][b+1] - psum[a][b+1] - psum[b+1][a] + psum[a][a]

print("prefix sum ready")
print("sanity check dsum(0, N-1) == dist_mat.sum():", abs(dsum(0, N-1) - dist_mat.sum()) < 1e-4)

prefix sum ready
sanity check dsum(0, N-1) == dist_mat.sum(): True


In [ ]:
# main DP  (H_nrm cost function from the paper)
# C[n,k,p] = min cost of grouping shots n..N-1 into k scenes
#            given accumulated squared-area p from shots 0..n-1
# J[n,k,p] = optimal last shot (0-indexed) of current group
# P[n,k,p] = total squared area of optimal solution from n onward
# all shot indices are 0-indexed

K = K_est
print(f"DP: N={N} shots  K={K} scenes")
print("this might take a while...")

C    = {}
J_dp = {}
P_dp = {}

# base case k=1: last single group covers shots n..N-1
k = 1
for n in range(N):
    area = (N - n) ** 2
    for p in area_sums(n, K-k):
        C[(n, k,p)]    = dsum(n, N-1)/(p + area)
        J_dp[(n,k, p)] = N- 1
        P_dp[(n,k, p)] = area

# fill k = 2..K
for k in range(2, K + 1):
    print(f"k={k}")
    for n in range(N - 1):
        if (N - n) < k:
            continue
        for p in area_sums(n, K - k):
            best_cost = float('inf')
            best_i= -1
            for i in range(n, N - 1):
                if (N - i) < k:
                    continue
                area = (i - n + 1) ** 2
                keyNext =(i + 1, k - 1, p + area)
                nextArea = P_dp.get(keyNext, 0)
                g = dsum(n, i) / (p + area + nextArea)
                cost= g + C.get(keyNext, 0)
                if cost < best_cost:
                    best_cost =cost
                    best_i = i
            if best_i >= 0:
                C[(n,k, p)] = best_cost
                J_dp[(n,k, p)]= best_i
                P_dp[(n,k, p)]= (best_i - n + 1)**2 + P_dp.get((best_i + 1, k-1, p + (best_i - n + 1)**2), 0)


DP: N=36 shots  K=2 scenes
this might take a while...
k=2


In [ ]:
t = []
n = 0
p_acc = 0
for step in range(K):
    key = (n, K - step, p_acc)
    if key not in J_dp:
        print(f"WARNING: key missing in DP table: {key}")
        break
    last = J_dp[key]
    t.append((n, last))
    p_acc += (last - n + 1) ** 2
    n = last + 1

predicted_scenes = t

print("Detected scenes (scene_num, start_frame, end_frame):")
for sc in predicted_scenes:
    print(sc)

Detected scenes (scene_num, start_frame, end_frame):
(1, 0, 1413)
(2, 1422, 2528)


In [ ]:
from Expirements.SceneSegmentation.utils import bgr_to_hsv, calc_hist, calculate_interval_metric


for vid, ann in zip(video_paths, gt_files):
    print(f"video {vid} \n")

    # shot detection
    cap = cv2.VideoCapture(vid)
    prev_hsv = None
    cuts = [0]; fi = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.resize(frame, (160, 90))
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        if prev_hsv is not None:
            diff = np.abs(hsv.astype("int32") - prev_hsv.astype("int32")).mean()
            if diff > THRESH: cuts.append(fi)
        prev_hsv = hsv.copy()
        fi += 1
    cap.release()
    cuts.append(fi)
    segs = [(cuts[i], cuts[i+1]-1) for i in range(len(cuts)-1)]
    segs = [s for s in segs if s[1]-s[0] >= MIN_LEN]
    N = len(segs)
    print(f"shots: {N}")

    # feature extraction
    cap2 = cv2.VideoCapture(vid)
    feats = []
    for st, en in segs:
        mid = (st+en)//2
        cap2.set(cv2.CAP_PROP_POS_FRAMES, mid)
        ok, frm = cap2.read()
        if not ok:
            feats.append(np.zeros(H_BINS+S_BINS+V_BINS, np.float32)); continue
        hsv_f = bgr_to_hsv(frm)
        feat = np.concatenate([
            calc_hist(hsv_f[...,0], H_BINS, (0,180)),
            calc_hist(hsv_f[...,1], S_BINS, (0,256)),
            calc_hist(hsv_f[...,2], V_BINS, (0,256))
        ])
        feat = feat/(feat.sum()+1e-9)
        feats.append(feat.astype(np.float32))
    cap2.release()
    feats = np.array(feats)

    # dist matrix
    dist_mat = np.zeros((N,N))
    for i in range(N):
        for j in range(N):
            dist_mat[i,j] = np.sqrt(np.sum((feats[i]-feats[j])**2))
    dist_mat = (dist_mat+dist_mat.T)/2.0

    # K estimation
    sv = np.linalg.svd(dist_mat, compute_uv=False)
    sv_log = np.log(sv[:int(len(sv)*0.5)]+1e-9)
    c0 = sv_log[0]
    sl = (sv_log[-1]-sv_log[0])/(len(sv_log)-1+1e-9)
    K_est = 2; max_d = 0
    for idx in range(len(sv_log)):
        d = abs(-sv_log[idx]+c0+sl*idx)/math.sqrt(sl**2+1)
        if d > max_d: max_d = d; K_est = idx
    K = max(min(K_est, N-1), 2)
    print(f"K={K}")

    # prefix sum
    ps = np.zeros((N+1,N+1))
    for ii in range(1,N+1):
        for jj in range(1,N+1):
            ps[ii][jj] = dist_mat[ii-1][jj-1]+ps[ii-1][jj]+ps[ii][jj-1]-ps[ii-1][jj-1]
    def ds(a,b):
        return ps[b+1][b+1]-ps[a][b+1]-ps[b+1][a]+ps[a][a]

    # DP
    C = {}; J = {}; P = {}
    k = 1
    for n in range(N):
        area = (N-n)**2
        for p in area_sums(n, K-k):
            C[(n,k,p)] = ds(n,N-1)/(p+area)
            J[(n,k,p)] = N-1
            P[(n,k,p)] = area
    for k in range(2, K+1):
        for n in range(N-1):
            if (N-n) < k: continue
            for p in area_sums(n, K-k):
                best = float('inf'); bi = -1
                for i in range(n, N-1):
                    if (N-i) < k: continue
                    area = (i-n+1)**2
                    kn = (i+1, k-1, p+area)
                    g = ds(n,i)/(p+area+P.get(kn,0))
                    cost = g+C.get(kn,0)
                    if cost < best: best = cost; bi = i
                if bi >= 0:
                    C[(n,k,p)] = best
                    J[(n,k,p)] = bi
                    P[(n,k,p)] = (bi-n+1)**2+P.get((bi+1,k-1,p+(bi-n+1)**2),0)

    # traceback
    t = []; n = 0; p_acc = 0
    for step in range(K):
        key = (n, K-step, p_acc)
        if key not in J: print(f"WARNING missing {key}"); break
        last = J[key]
        t.append((n,last))
        p_acc += (last-n+1)**2
        n = last+1

    pred_intervals = [(segs[f][0], segs[l][1]) for f,l in t if f < len(segs) and l < len(segs)]

    gt_intervals = []
    with open(ann) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2: gt_intervals.append((int(parts[0]), int(parts[1])))

    print(f"pred: {len(pred_intervals)}  gt: {len(gt_intervals)}")
    for m in ['precision', 'recall', 'f1', 'iou']:
        print(f"  {m}: {calculate_interval_metric([gt_intervals], [pred_intervals], m):.3f}")
    print()

video ./dataset/videos/Lord_Meia.mp4 

shots: 36
K=2
(1422, 2528)
pred: 2  gt: 27
  precision: 0.068
  recall: 0.044
  f1: 0.051
  iou: 0.043

video ./dataset/videos/La_Chute_dune_Plume.mp4 

shots: 121
K=13
(13248, 15585)
pred: 13  gt: 10
  precision: 0.490
  recall: 0.785
  f1: 0.436
  iou: 0.288

video ./dataset/videos/tears_of_steel_1080p.mp4 

shots: 153
K=13
(14139, 17619)
pred: 13  gt: 11
  precision: 0.733
  recall: 0.788
  f1: 0.649
  iou: 0.515

